# Klasifikasi Penyakit Daun Jagung dengan Fitur GLCM dan Warna HSV menggunakan K-Nearest Neighbor dan Random Forest

**Alur penelitian (sesuai tahapan machine learning):**

1. **Preprocessing**: pembersihan data (file rusak, duplikat), **pembagian 80:20 ke folder `latih` dan `uji`**, segmentasi daun, grayscale, **ekstraksi fitur tekstur GLCM + warna HSV** (citra → tabel), cek missing value & duplikat, split 80:20, normalisasi
2. **Modeling**: K-NN dan Random Forest pada dua skenario fitur (GLCM saja vs GLCM+HSV), parameter terbaik dicari dengan *5-fold cross validation* pada data latih
3. **Evaluasi**: confusion matrix, accuracy, precision, recall, F1-score pada data uji, uji McNemar, interval kepercayaan 95%, latensi & ukuran model
4. **Web demo**: model disimpan (`.pkl`) dan dipaketkan menjadi web Streamlit

**Dataset**: citra daun jagung dari PlantVillage, 4 kelas: bercak daun kelabu (*Cercospora*), karat daun (*common rust*), hawar daun (*northern leaf blight*), dan sehat. Dataset **diunduh otomatis** dari GitHub di sel 1 (hanya folder jagung), jadi tidak perlu unduh manual.

**Cara menjalankan:**
1. Runtime **CPU sudah cukup** (tidak perlu GPU, karena tidak memakai deep learning).
2. Pilih **Runtime → Run all**. Saat diminta, izinkan akses Google Drive (hasil disimpan di sana).

Semua tabel dan gambar untuk artikel otomatis tersimpan di folder `MyDrive/hasil_jagung_glcm`, termasuk `dataset_latih_uji.zip` (folder latih & uji).

> Sitasi dataset: Hughes, D. P., & Salathé, M. (2015). *An open access repository of images on plant health to enable the development of mobile disease diagnostics*. arXiv:1511.08060. Repositori: https://github.com/spMohanty/PlantVillage-Dataset

## 0. Konfigurasi

In [ ]:
import os, sys, json, time, hashlib, zipfile, shutil, pathlib, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

REPO_URL = 'https://github.com/spMohanty/PlantVillage-Dataset.git'
DATA_DIR = '/content/PlantVillage'                        # tempat dataset diunduh
APP_DIR  = '/content/web_app_jagung'                      # folder web app (dibuat otomatis)
SAVE_DIR = '/content/drive/MyDrive/hasil_jagung_glcm'     # semua hasil disimpan di Drive

# nama folder asli PlantVillage -> label yang dipakai di penelitian
KELAS_MAP = {
    'Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot': 'bercak_kelabu',
    'Corn_(maize)___Common_rust_':                        'karat_daun',
    'Corn_(maize)___Northern_Leaf_Blight':                'hawar_daun',
    'Corn_(maize)___healthy':                             'sehat',
}
NAMA_KELAS = {   # nama lengkap untuk tabel, grafik, dan web
    'bercak_kelabu': 'Bercak daun kelabu (Cercospora zeae-maydis)',
    'karat_daun':    'Karat daun (Puccinia sorghi)',
    'hawar_daun':    'Hawar daun (Exserohilum turcicum)',
    'sehat':         'Sehat',
}
KELAS = list(KELAS_MAP.values())
SEED = 42            # agar hasil bisa diulang (tulis di metodologi artikel)
TEST_SIZE = 0.20     # 80% latih : 20% uji
CV_FOLDS = 5         # 5-fold cross validation (pengganti data validasi)
REMOVE_NEAR_DUPLICATES = True   # hapus juga gambar yang HAMPIR sama
HAMMING_MAX = 3      # batas kemiripan near-duplicate (dari 256 bit, makin kecil makin ketat)

EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff', '.webp'}
for d in [SAVE_DIR, APP_DIR, os.path.join(APP_DIR, 'models')]:
    os.makedirs(d, exist_ok=True)
random.seed(SEED)
np.random.seed(SEED)

def simpan_gambar(nama):
    plt.savefig(os.path.join(SAVE_DIR, nama), dpi=200, bbox_inches='tight')

print('Hasil akan disimpan di:', SAVE_DIR)

## 1. Unduh dan kumpulkan data
Hanya folder jagung pada `raw/color` yang diunduh dari repositori PlantVillage (sekitar 3.800 citra, beberapa detik). Label diambil dari **nama folder**.

In [ ]:
import subprocess
if not os.path.isdir(os.path.join(DATA_DIR, 'raw', 'color')):
    shutil.rmtree(DATA_DIR, ignore_errors=True)
    subprocess.run(['git', 'clone', '-q', '--depth', '1', '--filter=blob:none', '--no-checkout',
                    REPO_URL, DATA_DIR], check=True)
    subprocess.run(['git', '-C', DATA_DIR, 'sparse-checkout', 'set', '--no-cone', '/raw/color/Corn_*'],
                   check=True)
    subprocess.run(['git', '-C', DATA_DIR, 'checkout', '-q'], check=True)

rows = []
for folder, label in KELAS_MAP.items():
    d = pathlib.Path(DATA_DIR, 'raw', 'color', folder)
    for p in sorted(d.iterdir()):
        if p.is_file() and p.suffix.lower() in EXTS and not p.name.startswith('.'):
            rows.append({'path': str(p), 'label': label, 'folder_asli': folder})
data = pd.DataFrame(rows)
assert len(data) > 0, 'Dataset tidak ditemukan. Jalankan ulang sel ini.'
tabel_awal = data.groupby(['folder_asli', 'label']).size().rename('Jumlah citra').reset_index()
display(tabel_awal)
tabel_awal.to_csv(os.path.join(SAVE_DIR, 'tabel_dataset_awal.csv'), index=False)
print('Total citra:', len(data))

## 2. Pembersihan data (data cleaning)
Agar dataset "bersih":
1. **File rusak** → dibuang.
2. **Format** dicatat (mode warna & ukuran) lalu diseragamkan: RGB 3 kanal, 256×256.
3. **Duplikat persis** (isi piksel identik, dicek dengan hash MD5) → disimpan satu saja.
4. **Duplikat hampir sama** (dicek dengan *difference hash* 256 bit) → disimpan satu saja.
5. Jika satu kelompok duplikat punya **label berbeda**, semuanya dibuang karena labelnya bertentangan.

In [ ]:
def dhash_bits(img, size=16):
    g = np.asarray(img.convert('L').resize((size + 1, size), Image.BILINEAR), dtype=np.int16)
    return (g[:, 1:] > g[:, :-1]).ravel()

info, hashes = [], []
for path in data['path']:
    rec = {'path': path, 'rusak': False, 'mode': None, 'ukuran': None, 'md5': None}
    try:
        with Image.open(path) as im:
            im.verify()                                # cek struktur file
        with Image.open(path) as im:
            rec['mode'], rec['ukuran'] = im.mode, f'{im.size[0]}x{im.size[1]}'
            rgb = im.convert('RGB')
            rgb.load()
        rec['md5'] = hashlib.md5(rgb.tobytes()).hexdigest()
        hashes.append(dhash_bits(rgb))
    except Exception as e:
        rec['rusak'] = True
        hashes.append(None)
    info.append(rec)
info = pd.DataFrame(info)
data = data.merge(info, on='path')
print('File rusak:', int(data['rusak'].sum()))
print('\nMode warna:'); print(data.loc[~data['rusak'], 'mode'].value_counts().to_string())
print('\nUkuran gambar:'); print(data.loc[~data['rusak'], 'ukuran'].value_counts().head(5).to_string())

In [ ]:
# ---- Kelompokkan duplikat (union-find) ----
n = len(data)
induk = list(range(n))
def cari(i):
    while induk[i] != i:
        induk[i] = induk[induk[i]]
        i = induk[i]
    return i
def gabung(a, b):
    ra, rb = cari(a), cari(b)
    if ra != rb:
        induk[max(ra, rb)] = min(ra, rb)

valid = np.where(~data['rusak'].to_numpy())[0]

# 1) duplikat persis: md5 piksel sama
n_pasangan_persis = 0
for _, idx in data.loc[valid].groupby('md5').groups.items():
    idx = list(idx)
    for j in idx[1:]:
        gabung(idx[0], j)
        n_pasangan_persis += 1

# 2) duplikat hampir sama: jarak Hamming dHash <= HAMMING_MAX
pasangan_mirip = []
if REMOVE_NEAR_DUPLICATES and len(valid) > 1:
    B = np.stack([hashes[i] for i in valid]).astype(np.float32)
    jml = B.sum(1)
    for s in range(0, len(valid), 1000):
        blok = B[s:s + 1000]
        jarak = jml[s:s + 1000, None] + jml[None, :] - 2 * (blok @ B.T)
        ii, jj = np.where(jarak <= HAMMING_MAX)
        for a, b in zip(ii + s, jj):
            if a < b:
                i, j = valid[a], valid[b]
                if data.at[i, 'md5'] != data.at[j, 'md5']:
                    pasangan_mirip.append((i, j))
                gabung(i, j)

data['grup'] = [cari(i) for i in range(n)]
data['hapus'] = ''
data.loc[data['rusak'], 'hapus'] = 'file rusak'
for g, idx in data.loc[~data['rusak']].groupby('grup').groups.items():
    idx = sorted(idx)
    if len(idx) == 1:
        continue
    if data.loc[idx, 'label'].nunique() > 1:
        data.loc[idx, 'hapus'] = 'duplikat beda label'
    else:
        data.loc[idx[1:], 'hapus'] = 'duplikat (disimpan satu)'

print('Pasangan duplikat persis      :', n_pasangan_persis)
print('Pasangan duplikat hampir sama :', len(pasangan_mirip))

In [ ]:
# Tampilkan contoh pasangan hampir sama untuk dicek manual
if pasangan_mirip:
    k = min(5, len(pasangan_mirip))
    fig, ax = plt.subplots(k, 2, figsize=(5, 2.5 * k), squeeze=False)
    for r, (i, j) in enumerate(pasangan_mirip[:k]):
        for c, t in enumerate([i, j]):
            ax[r, c].imshow(Image.open(data.at[t, 'path']).convert('RGB'))
            ax[r, c].set_title(f"{data.at[t, 'label']} | {pathlib.Path(data.at[t, 'path']).name}", fontsize=7)
            ax[r, c].axis('off')
    plt.suptitle('Contoh pasangan duplikat hampir sama')
    plt.tight_layout(); simpan_gambar('contoh_near_duplicate.png'); plt.show()
else:
    print('Tidak ada duplikat hampir sama.')

In [ ]:
ringkas = [('Gambar awal', len(data))]
sisa = len(data)
for alasan in ['file rusak', 'duplikat (disimpan satu)', 'duplikat beda label']:
    jml = int((data['hapus'] == alasan).sum())
    sisa -= jml
    ringkas.append((f'Dihapus: {alasan}', jml))
ringkas.append(('Gambar bersih', sisa))
ringkasan_cleaning = pd.DataFrame(ringkas, columns=['Tahap', 'Jumlah'])
display(ringkasan_cleaning)
ringkasan_cleaning.to_csv(os.path.join(SAVE_DIR, 'tabel_pembersihan_data.csv'), index=False)

bersih = data[data['hapus'] == ''].reset_index(drop=True)
dist = bersih['label'].value_counts().reindex(KELAS)
display(dist.rename('Jumlah citra').to_frame())

plt.figure(figsize=(6, 3.5))
plt.bar(dist.index, dist.values, color='#4C72B0')
for x, y in zip(dist.index, dist.values):
    plt.text(x, y, str(y), ha='center', va='bottom')
plt.title('Distribusi kelas setelah pembersihan'); plt.ylabel('Jumlah citra')
plt.tight_layout(); simpan_gambar('distribusi_kelas.png'); plt.show()

## 3. Pembagian data 80:20 ke folder `latih` dan `uji`
Dataset PlantVillage tidak menyediakan pembagian bawaan, jadi citra yang **sudah bersih** dibagi di sini:
- **80% data latih** dan **20% data uji**, dipilih acak **per kelas** (*stratified*) dengan `random_state = 42`, sehingga proporsi tiap kelas sama dan hasilnya selalu sama setiap kali dijalankan.
- Citra **disalin** ke folder baru (folder asli tidak diubah):

```
PlantVillage/raw/color/
├── latih/  ── bercak_kelabu/  karat_daun/  hawar_daun/  sehat/
└── uji/    ── bercak_kelabu/  karat_daun/  hawar_daun/  sehat/
```
- Model **hanya belajar dari folder `latih`**. Pemilihan parameter memakai *5-fold cross validation* di dalam data latih (pengganti data validasi). Folder `uji` baru dipakai **sekali** di tahap evaluasi.
- Kedua folder juga disimpan sebagai `dataset_latih_uji.zip` di Google Drive, sehingga bisa dibuka di komputer atau dipakai untuk menguji web demo dengan citra yang belum pernah dilihat model.

In [ ]:
from sklearn.model_selection import train_test_split

COLOR_DIR = os.path.join(DATA_DIR, 'raw', 'color')
idx_latih, idx_uji = train_test_split(bersih.index, test_size=TEST_SIZE,
                                      stratify=bersih['label'], random_state=SEED)
bersih['subset'] = 'latih'
bersih.loc[idx_uji, 'subset'] = 'uji'

# Salin citra ke folder latih/ dan uji/ (dihapus dulu jika sel ini dijalankan ulang)
for sub in ['latih', 'uji']:
    shutil.rmtree(os.path.join(COLOR_DIR, sub), ignore_errors=True)
path_baru = []
for _, r in bersih.iterrows():
    tujuan = os.path.join(COLOR_DIR, r['subset'], r['label'], pathlib.Path(r['path']).name)
    os.makedirs(os.path.dirname(tujuan), exist_ok=True)
    shutil.copy2(r['path'], tujuan)
    path_baru.append(tujuan)
bersih['path_asli'] = bersih['path']
bersih['path'] = path_baru          # tahap berikutnya membaca citra dari folder latih/uji

# Cek: hitung isi folder yang benar-benar terbentuk
tabel_split = pd.DataFrame({
    sub: {k: len(os.listdir(os.path.join(COLOR_DIR, sub, k))) for k in KELAS}
    for sub in ['latih', 'uji']}).rename(columns={'latih': 'Latih', 'uji': 'Uji'})
tabel_split.loc['Total'] = tabel_split.sum()
assert tabel_split.loc['Total'].sum() == len(bersih), 'Jumlah file di folder tidak sama dengan data bersih'
display(tabel_split)
tabel_split.to_csv(os.path.join(SAVE_DIR, 'tabel_pembagian_data.csv'))

# Daftar nama file per subset (untuk lampiran / bukti pembagian data)
daftar = bersih[['subset', 'label', 'path']].copy()
daftar['file'] = daftar['path'].map(lambda p: pathlib.Path(p).name)
daftar['folder'] = daftar['path'].map(lambda p: os.path.relpath(os.path.dirname(p), DATA_DIR))
daftar[['subset', 'label', 'file', 'folder']].sort_values(['subset', 'label', 'file']) \
    .to_csv(os.path.join(SAVE_DIR, 'daftar_pembagian_data.csv'), index=False)

# Simpan folder latih/ dan uji/ sebagai zip di Google Drive
zip_data = os.path.join(SAVE_DIR, 'dataset_latih_uji.zip')
with zipfile.ZipFile(zip_data, 'w', zipfile.ZIP_STORED) as zf:   # JPG sudah terkompresi
    for p in path_baru:
        zf.write(p, os.path.join('dataset_latih_uji', os.path.relpath(p, COLOR_DIR)))
print(f'Folder dibuat di : {COLOR_DIR}/latih dan {COLOR_DIR}/uji')
print(f'Zip tersimpan di : {zip_data} ({os.path.getsize(zip_data) / 1e6:.1f} MB)')

## 4. Preprocessing dan segmentasi
Modul `glcm_features.py` berisi seluruh preprocessing dan ekstraksi fitur. File yang **sama** nanti dipakai web app, sehingga gambar yang diunggah diproses persis seperti data latih.

Tahapan per gambar: RGB 256×256 → **segmentasi daun** → fitur tekstur GLCM (grayscale) dan fitur warna HSV, keduanya **hanya pada area daun**.

**Kenapa segmentasi penting di dataset ini?** Latar foto PlantVillage berbeda antar kelas: sebagian besar foto karat daun berlatar hitam, kelas lain berlatar abu-abu, dan daun sehat banyak yang difoto close-up memenuhi bingkai. Tanpa segmentasi, model bisa "belajar" dari warna latar, bukan dari penyakitnya. Segmentasi berbasis warna (HSV): latar = piksel hitam (V < 0,12) atau abu-abu (saturasi S < 0,12) yang tersambung ke tepi gambar; bercak pucat di tengah daun tetap dihitung sebagai daun.

In [ ]:
%%writefile /content/web_app_jagung/glcm_features.py
"""
glcm_features.py
Preprocessing + ekstraksi fitur tekstur GLCM dan warna HSV untuk citra daun jagung.

File ini dipakai di DUA tempat: notebook (saat melatih model) dan web app
(saat memprediksi). Karena kodenya sama persis, gambar yang diunggah ke web
diproses dengan cara yang identik dengan data latih.
"""
import numpy as np
from PIL import Image
from scipy import ndimage as ndi
from skimage.color import rgb2hsv
from skimage.feature import graycomatrix, graycoprops

IMG_SIZE = 256                      # ukuran citra PlantVillage (256 x 256)
LEVELS = 64                         # jumlah tingkat keabuan GLCM (256 -> 64)
DISTANCES = [1, 2, 3]               # jarak piksel bertetangga
ANGLES = [0, np.pi / 4, np.pi / 2, 3 * np.pi / 4]   # 0, 45, 90, 135 derajat
PROPS = ['contrast', 'dissimilarity', 'homogeneity', 'energy', 'ASM', 'correlation']

BATAS_GELAP = 0.12        # V < 0.12 -> latar hitam
BATAS_SATURASI = 0.12     # S < 0.12 -> latar abu-abu/putih (daun jagung S > 0.15)
LUAS_MIN = 0.01           # potongan daun < 1% luas citra dibuang (noise)


WARNA = [f'{k}_{m}' for k in ('H', 'S', 'V') for m in ('mean', 'std', 'skew')]


def feature_names(distances=DISTANCES):
    """Nama kolom fitur GLCM, contoh: contrast_d1, entropy_d3."""
    return [f'{p}_d{d}' for d in distances for p in PROPS + ['entropy']]


def all_feature_names():
    """21 fitur GLCM + 9 fitur warna HSV (urutan sama dengan extract_features)."""
    return feature_names() + WARNA


def load_rgb(source, size=IMG_SIZE):
    """Buka gambar (path / file upload / PIL.Image) -> array RGB uint8 size x size."""
    img = source if isinstance(source, Image.Image) else Image.open(source)
    img = img.convert('RGB')                      # buang kanal alpha / grayscale -> 3 kanal
    if img.size != (size, size):
        img = img.resize((size, size), Image.BILINEAR)
    return np.asarray(img, dtype=np.uint8)


def to_gray(rgb):
    """RGB -> grayscale (luminance ITU-R BT.601)."""
    rgb = rgb.astype(np.float64)
    return 0.299 * rgb[..., 0] + 0.587 * rgb[..., 1] + 0.114 * rgb[..., 2]


def _disk(r):
    y, x = np.ogrid[-r:r + 1, -r:r + 1]
    return x * x + y * y <= r * r


def segment_leaf(rgb):
    """
    Segmentasi daun berbasis warna (HSV).
    Latar = piksel hitam (V kecil) atau abu-abu (saturasi S kecil) YANG TERSAMBUNG
    ke tepi gambar. Bagian pucat di tengah daun (bercak, pantulan cahaya) tetap
    dihitung sebagai daun. Mengembalikan (mask_daun, berhasil). Jika tidak ada
    latar (daun memenuhi bingkai), seluruh citra dianggap daun.
    """
    hsv = rgb2hsv(rgb)
    S, V = hsv[..., 1], hsv[..., 2]
    kandidat = (V < BATAS_GELAP) | (S < BATAS_SATURASI)
    lab, _ = ndi.label(kandidat)
    tepi = np.unique(np.concatenate([lab[0, :], lab[-1, :], lab[:, 0], lab[:, -1]]))
    latar = np.isin(lab, tepi[tepi > 0])
    daun = ndi.binary_opening(~latar, structure=_disk(2))
    daun = ndi.binary_fill_holes(daun)
    lab, n = ndi.label(daun)
    if n > 0:
        luas = np.bincount(lab.ravel())
        luas[0] = 0
        daun = np.isin(lab, np.where(luas >= LUAS_MIN * daun.size)[0])
    frac = daun.mean()
    if frac < 0.05:                               # segmentasi gagal
        return np.ones(S.shape, bool), False
    if frac > 0.97:                               # daun memenuhi bingkai
        return np.ones(S.shape, bool), True
    return daun, True


def glcm_features(gray, mask, distances=DISTANCES, levels=LEVELS):
    """
    Hitung fitur GLCM HANYA pada area daun (latar diabaikan).
    Untuk tiap jarak: 7 fitur dirata-rata dari 4 sudut.
    """
    # Kuantisasi 0..255 -> 1..levels-1; nilai 0 dicadangkan untuk latar
    q = np.floor(np.clip(gray, 0, 255) / 256.0 * (levels - 1)).astype(np.uint8) + 1
    q[~mask] = 0
    P = graycomatrix(q, distances=distances, angles=ANGLES, levels=levels,
                     symmetric=True, normed=False)
    P = P[1:, 1:, :, :].astype(np.float64)        # buang pasangan yang melibatkan latar

    fitur = []
    for i in range(len(distances)):
        Pd = P[:, :, i:i + 1, :]
        total = Pd.sum(axis=(0, 1), keepdims=True)
        total[total == 0] = 1.0
        Pn = Pd / total
        nilai = {p: float(graycoprops(Pn, p)[0].mean()) for p in PROPS}
        ent = []
        for a in range(Pn.shape[3]):
            p = Pn[:, :, 0, a]
            p = p[p > 0]
            ent.append(float(-(p * np.log2(p)).sum()))
        nilai['entropy'] = float(np.mean(ent))
        fitur.extend(nilai[k] for k in PROPS + ['entropy'])
    return np.asarray(fitur, dtype=np.float64)


def color_features(rgb, mask):
    """Momen warna HSV pada area daun: mean, standar deviasi, skewness tiap kanal."""
    hsv = rgb2hsv(rgb)
    fitur = []
    for c in range(3):
        v = hsv[..., c][mask].astype(np.float64)
        mu, sd = v.mean(), v.std()
        sk = float(np.mean((v - mu) ** 3) / sd ** 3) if sd > 1e-12 else 0.0
        fitur += [float(mu), float(sd), sk]
    return np.asarray(fitur, dtype=np.float64)


def extract_features(source):
    """Pipeline lengkap: buka -> RGB 256x256 -> segmentasi daun -> GLCM (grayscale) + warna HSV.
    Mengembalikan (vektor 30 fitur sesuai all_feature_names(), mask, segmentasi_berhasil)."""
    rgb = load_rgb(source)
    mask, ok = segment_leaf(rgb)
    f = np.concatenate([glcm_features(to_gray(rgb), mask), color_features(rgb, mask)])
    return f, mask, ok

In [ ]:
sys.path.insert(0, APP_DIR)
import importlib, glcm_features
importlib.reload(glcm_features)
from glcm_features import load_rgb, to_gray, segment_leaf, extract_features, \
    feature_names, all_feature_names

# Visualisasi tahapan preprocessing: 1 contoh per kelas (gambar untuk artikel)
fig, ax = plt.subplots(len(KELAS), 4, figsize=(10, 2.6 * len(KELAS)), squeeze=False)
for r, kls in enumerate(KELAS):
    path = bersih.loc[bersih['label'] == kls, 'path'].iloc[0]
    rgb = load_rgb(path); gray = to_gray(rgb); mask, ok = segment_leaf(rgb)
    seg = np.where(mask, gray, 0)
    for c, (im, judul) in enumerate([(rgb, 'Citra asli'), (gray, 'Grayscale'),
                                     (mask, 'Mask daun'), (seg, 'Area daun (GLCM)')]):
        if c == 0:
            ax[r, c].imshow(im)
        else:   # skala tetap agar mask yang seluruhnya daun tampil putih
            ax[r, c].imshow(im, cmap='gray', vmin=0, vmax=1 if c == 2 else 255)
        ax[r, c].set_title(f'{judul}\n({NAMA_KELAS[kls]})' if c == 0 else judul, fontsize=8)
        ax[r, c].axis('off')
plt.tight_layout(); simpan_gambar('tahapan_preprocessing.png'); plt.show()

## 5. Ekstraksi fitur (citra → tabel)
**Fitur tekstur GLCM (21 fitur)**: 64 tingkat keabuan, jarak 1, 2, 3 piksel, sudut 0°, 45°, 90°, 135°. Fitur *contrast, dissimilarity, homogeneity, energy, ASM, correlation, entropy* dirata-rata dari 4 sudut (agar tidak bergantung pada arah daun) → 7 fitur × 3 jarak.

**Fitur warna HSV (9 fitur)**: rata-rata, standar deviasi, dan *skewness* kanal H, S, V pada area daun (momen warna). GLCM bekerja pada citra grayscale sehingga tidak "melihat" warna; padahal lesi penyakit jagung dibedakan oleh warnanya (karat: cokelat kemerahan, hawar & bercak kelabu: cokelat pucat/keabuan).

Hasilnya adalah **data terstruktur**: satu baris per gambar, satu kolom per fitur, ditambah kolom label.

In [ ]:
from joblib import Parallel, delayed

def proses(path):
    f, mask, ok = extract_features(path)
    return f, ok

t0 = time.time()
hasil = Parallel(n_jobs=-1)(delayed(proses)(p) for p in bersih['path'])
FITUR_GLCM = feature_names()          # 21 fitur tekstur
FITUR_SEMUA = all_feature_names()     # 21 tekstur + 9 warna
df = pd.DataFrame([h[0] for h in hasil], columns=FITUR_SEMUA)
df.insert(0, 'file', [pathlib.Path(p).name for p in bersih['path']])
df.insert(1, 'path', bersih['path'].values)
df['segmentasi_ok'] = [h[1] for h in hasil]
df['label'] = bersih['label'].values
df['subset'] = bersih['subset'].values      # latih / uji (dari sel 3)
print(f'Ekstraksi selesai dalam {time.time() - t0:.1f} detik')
print('Segmentasi gagal (memakai seluruh citra):', int((~df['segmentasi_ok']).sum()))
df.to_csv(os.path.join(SAVE_DIR, 'dataset_fitur.csv'), index=False)
display(df.drop(columns=['path']).head())

## 6. Pemeriksaan data tabel
- **Missing value** (NaN / tak hingga) → harus 0
- **Baris fitur duplikat** → dibuang
- **Tipe data**: semua fitur numerik, jadi **tidak perlu encoding**. Label tetap berupa teks karena scikit-learn bisa langsung memakainya.

In [ ]:
X_all = df[FITUR_SEMUA]
print('Missing value (NaN) :', int(X_all.isna().sum().sum()))
print('Nilai tak hingga    :', int(np.isinf(X_all.to_numpy()).sum()))
print('Baris fitur duplikat:', int(X_all.duplicated().sum()))
print('Semua fitur numerik :', all(pd.api.types.is_numeric_dtype(X_all[c]) for c in FITUR_SEMUA))

buang = X_all.isna().any(axis=1) | np.isinf(X_all).any(axis=1) | X_all.duplicated()
df = df[~buang].reset_index(drop=True)
print('Baris dibuang:', int(buang.sum()), '| Sisa:', len(df))

statistik = df[FITUR_SEMUA].describe().T
statistik.to_csv(os.path.join(SAVE_DIR, 'statistik_deskriptif_fitur.csv'))
display(statistik.round(4))

## 7. Memisahkan tabel fitur: latih dan uji
Baris tabel dipisahkan sesuai **folder asal citranya** (sel 3): citra di folder `latih` → data latih, citra di folder `uji` → data uji.
Jumlah citra per kelas **tidak seimbang** (bercak kelabu paling sedikit), karena itu evaluasi memakai rata-rata *macro* (setiap kelas berbobot sama) dan Random Forest juga dicoba dengan `class_weight='balanced'`.

In [ ]:
latih = df['subset'] == 'latih'
X_train, y_train = df.loc[latih, FITUR_SEMUA], df.loc[latih, 'label']
X_test,  y_test  = df.loc[~latih, FITUR_SEMUA], df.loc[~latih, 'label']
print('Data latih:', len(X_train), '| Data uji:', len(X_test))
display(pd.DataFrame({'Latih': y_train.value_counts(), 'Uji': y_test.value_counts()}).reindex(KELAS))

## 8. Modeling
Dua skenario fitur × dua algoritma (data latih dan uji yang sama persis):

| Skenario | Fitur | Tujuan |
|---|---|---|
| **GLCM** | 21 fitur tekstur | sama dengan pendahulu (Purba dkk., 2025), sebagai pembanding |
| **GLCM+HSV** | 21 tekstur + 9 warna | skenario utama, dipakai di web demo |

- **K-NN**: fitur dinormalisasi Min-Max (wajib, karena K-NN berbasis jarak). Normalisasi ada di dalam *pipeline*, jadi hanya "belajar" dari data latih (tidak ada kebocoran data).
- **Random Forest**: tidak butuh normalisasi.

Parameter terbaik dicari dengan **GridSearchCV** (5-fold cross validation pada data latih). Sel ini butuh beberapa menit.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold

cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=SEED)
SKENARIO = {'GLCM': FITUR_GLCM, 'GLCM+HSV': FITUR_SEMUA}
UTAMA = 'GLCM+HSV'                       # skenario yang dipakai web demo
NAMA = {'knn': 'K-Nearest Neighbor', 'rf': 'Random Forest'}

def kandidat(key):
    if key == 'knn':
        return (Pipeline([('scaler', MinMaxScaler()), ('knn', KNeighborsClassifier())]),
                {'knn__n_neighbors': list(range(1, 32, 2)),
                 'knn__weights': ['uniform', 'distance'],
                 'knn__metric': ['euclidean', 'manhattan']})
    return (RandomForestClassifier(n_estimators=200, random_state=SEED, n_jobs=-1),
            {'max_depth': [None, 20],
             'min_samples_split': [2, 5],
             'max_features': ['sqrt', 'log2'],
             'class_weight': [None, 'balanced']})

grid, model_terbaik, waktu_latih = {}, {}, {}
for sk, kolom in SKENARIO.items():
    for key in NAMA:
        est, param = kandidat(key)
        print(f'[{sk}] mencari parameter terbaik {NAMA[key]} ...')
        t0 = time.time()
        g = GridSearchCV(est, param, cv=cv, scoring='accuracy', n_jobs=-1)
        g.fit(X_train[kolom], y_train)
        waktu_latih[sk, key] = time.time() - t0
        grid[sk, key], model_terbaik[sk, key] = g, g.best_estimator_
        print(f'    parameter terbaik  : {g.best_params_}')
        print(f'    akurasi CV (5-fold): {g.best_score_:.4f} | waktu: {waktu_latih[sk, key]:.1f} detik')

In [ ]:
# Grafik pengaruh nilai k (K-NN, skenario utama) terhadap akurasi CV -> bahan pembahasan
res = pd.DataFrame(grid[UTAMA, 'knn'].cv_results_)
bp = grid[UTAMA, 'knn'].best_params_
sel = res[(res['param_knn__weights'] == bp['knn__weights']) & (res['param_knn__metric'] == bp['knn__metric'])]
sel = sel.sort_values('param_knn__n_neighbors')
plt.figure(figsize=(7, 3.5))
plt.errorbar(sel['param_knn__n_neighbors'].astype(int), sel['mean_test_score'],
             yerr=sel['std_test_score'], marker='o', capsize=3)
plt.xlabel('Nilai k'); plt.ylabel('Akurasi CV (5-fold)')
plt.title(f"Pengaruh nilai k pada K-NN [{UTAMA}] (weights={bp['knn__weights']}, metric={bp['knn__metric']})")
plt.grid(alpha=.3); plt.tight_layout(); simpan_gambar('pengaruh_nilai_k.png'); plt.show()

# Feature importance Random Forest -> fitur mana yang paling berpengaruh
imp = pd.Series(model_terbaik[UTAMA, 'rf'].feature_importances_, index=SKENARIO[UTAMA]).sort_values()
warna_bar = ['#DD8452' if f in FITUR_GLCM else '#55A868' for f in imp.index]
plt.figure(figsize=(7, 8))
plt.barh(imp.index, imp.values, color=warna_bar)
plt.xlabel('Feature importance')
plt.title('Kepentingan fitur (Random Forest)\noranye = tekstur GLCM, hijau = warna HSV')
plt.tight_layout(); simpan_gambar('feature_importance_rf.png'); plt.show()
imp.sort_values(ascending=False).to_csv(os.path.join(SAVE_DIR, 'feature_importance_rf.csv'))

## 9. Evaluasi pada data uji
Metrik: **accuracy, precision, recall, F1-score** (rata-rata *macro*) dan **confusion matrix**, sesuai evaluasi tugas klasifikasi.

In [ ]:
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             classification_report, confusion_matrix, ConfusionMatrixDisplay)

ringkasan, prediksi = [], {}
for (sk, key), model in model_terbaik.items():
    t0 = time.time()
    y_pred = model.predict(X_test[SKENARIO[sk]])
    t_pred = time.time() - t0
    prediksi[sk, key] = y_pred
    ringkasan.append({
        'Fitur': sk, 'Model': NAMA[key],
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred, average='macro', zero_division=0),
        'Recall': recall_score(y_test, y_pred, average='macro', zero_division=0),
        'F1-score': f1_score(y_test, y_pred, average='macro', zero_division=0),
        'Akurasi CV': grid[sk, key].best_score_,
        'Waktu latih+tuning (s)': waktu_latih[sk, key],
        'Waktu prediksi (ms/citra)': 1000 * t_pred / len(X_test),
    })
    laporan = classification_report(y_test, y_pred, labels=KELAS, digits=4, zero_division=0)
    print(f'===== {NAMA[key]} | fitur {sk} =====\n{laporan}')
    nama_file = f"classification_report_{sk.lower().replace('+', '_')}_{key}.txt"
    with open(os.path.join(SAVE_DIR, nama_file), 'w') as f:
        f.write(f'{NAMA[key]} | fitur {sk}\nParameter terbaik: {grid[sk, key].best_params_}\n\n{laporan}')

tabel_hasil = pd.DataFrame(ringkasan).set_index(['Fitur', 'Model'])
tabel_hasil.to_csv(os.path.join(SAVE_DIR, 'tabel_perbandingan_model.csv'))
display(tabel_hasil.round(4))

In [ ]:
fig, ax = plt.subplots(len(SKENARIO), 2, figsize=(12, 5 * len(SKENARIO)), squeeze=False)
for r, sk in enumerate(SKENARIO):
    for c, key in enumerate(NAMA):
        a = ax[r, c]
        cm = confusion_matrix(y_test, prediksi[sk, key], labels=KELAS)
        ConfusionMatrixDisplay(cm, display_labels=KELAS).plot(ax=a, cmap='Blues', colorbar=False)
        a.set_title(f'{NAMA[key]} | fitur {sk} (akurasi {accuracy_score(y_test, prediksi[sk, key]):.2%})')
        a.set_xlabel('Prediksi'); a.set_ylabel('Label asli')
plt.tight_layout(); simpan_gambar('confusion_matrix.png'); plt.show()

metrik = ['Accuracy', 'Precision', 'Recall', 'F1-score']
baris = list(tabel_hasil.index)
lebar = 0.8 / len(baris)
xpos = np.arange(len(metrik))
plt.figure(figsize=(10, 4.5))
for i, idx in enumerate(baris):
    b = plt.bar(xpos + (i - (len(baris) - 1) / 2) * lebar, tabel_hasil.loc[idx, metrik].values,
                lebar, label=f'{idx[1]} ({idx[0]})')
    plt.bar_label(b, fmt='%.3f', fontsize=7)
plt.xticks(xpos, metrik); plt.ylim(0, 1.1); plt.ylabel('Nilai')
plt.title('Perbandingan kinerja K-NN dan Random Forest pada dua skenario fitur')
plt.legend(loc='lower right', fontsize=8)
plt.tight_layout(); simpan_gambar('perbandingan_metrik.png'); plt.show()

## 10. Uji signifikansi statistik dan kinerja sistem
Melengkapi syarat materi Pertemuan 4–5 (validasi statistik & metrik sistem ter-*deploy*):
- **Uji McNemar**: apakah selisih akurasi dua model pada data uji yang **sama** signifikan (p < 0,05)?
- **Interval kepercayaan 95% (bootstrap, 1.000 ulangan)** untuk accuracy dan F1 *macro*.
- **Latensi inferensi** per citra (baca gambar → segmentasi → fitur → prediksi) dan **ukuran file model**.
- **Lingkungan komputasi** (versi Python & library) untuk bagian *reproducibility* di artikel.

In [ ]:
from scipy.stats import binomtest, chi2

y_arr = y_test.to_numpy()
def mcnemar(pred_a, pred_b):
    a_ok, b_ok = pred_a == y_arr, pred_b == y_arr
    b = int((a_ok & ~b_ok).sum())          # A benar, B salah
    c = int((~a_ok & b_ok).sum())          # A salah, B benar
    stat = (abs(b - c) - 1) ** 2 / (b + c) if b + c else 0.0
    p_exact = binomtest(b, b + c, 0.5).pvalue if b + c else 1.0
    return {'b (A benar, B salah)': b, 'c (A salah, B benar)': c,
            'chi2 (koreksi kontinuitas)': stat, 'p-value': chi2.sf(stat, 1), 'p-value eksak': p_exact}

pasangan = {
    'K-NN vs RF (GLCM+HSV)': (prediksi['GLCM+HSV', 'knn'], prediksi['GLCM+HSV', 'rf']),
    'K-NN vs RF (GLCM)': (prediksi['GLCM', 'knn'], prediksi['GLCM', 'rf']),
    'K-NN: GLCM vs GLCM+HSV': (prediksi['GLCM', 'knn'], prediksi['GLCM+HSV', 'knn']),
    'RF: GLCM vs GLCM+HSV': (prediksi['GLCM', 'rf'], prediksi['GLCM+HSV', 'rf']),
}
tabel_mcnemar = pd.DataFrame({k: mcnemar(*v) for k, v in pasangan.items()}).T
tabel_mcnemar['Signifikan (p<0,05)'] = tabel_mcnemar['p-value'] < 0.05
tabel_mcnemar.to_csv(os.path.join(SAVE_DIR, 'uji_mcnemar.csv'))
display(tabel_mcnemar)

# Bootstrap 95% CI: data uji di-resample dengan pengembalian sebanyak 1.000 kali
rng = np.random.default_rng(SEED)
idx_bs = rng.integers(0, len(y_arr), (1000, len(y_arr)))
baris_ci = []
for (sk, key), p in prediksi.items():
    acc = [(p[i] == y_arr[i]).mean() for i in idx_bs]
    f1m = [f1_score(y_arr[i], p[i], average='macro') for i in idx_bs]
    baris_ci.append({'Fitur': sk, 'Model': NAMA[key],
                     'Accuracy': accuracy_score(y_arr, p), 'Acc CI 2,5%': np.percentile(acc, 2.5), 'Acc CI 97,5%': np.percentile(acc, 97.5),
                     'F1 macro': f1_score(y_arr, p, average='macro'), 'F1 CI 2,5%': np.percentile(f1m, 2.5), 'F1 CI 97,5%': np.percentile(f1m, 97.5)})
tabel_ci = pd.DataFrame(baris_ci).set_index(['Fitur', 'Model'])
tabel_ci.to_csv(os.path.join(SAVE_DIR, 'interval_kepercayaan_bootstrap.csv'))
display(tabel_ci.round(4))

In [ ]:
import platform, tempfile, joblib, sklearn, skimage, scipy

# Latensi end-to-end per citra (200 citra uji pertama), sama dengan alur web demo
jalur_uji = df.loc[~latih, 'path'].tolist()[:200]
for p in jalur_uji[:5]:
    extract_features(p)                                      # pemanasan
t_fitur, X_lat = [], []
for p in jalur_uji:
    t0 = time.perf_counter(); f, _, _ = extract_features(p); t_fitur.append(time.perf_counter() - t0); X_lat.append(f)
X_lat = pd.DataFrame(X_lat, columns=FITUR_SEMUA)
baris_sistem = []
for key in NAMA:
    m = model_terbaik[UTAMA, key]
    if key == 'rf':
        m.set_params(n_jobs=1)                               # satu permintaan = satu proses
    m.predict(X_lat.iloc[:1])
    t_pred = []
    for i in range(len(X_lat)):
        t0 = time.perf_counter(); m.predict(X_lat.iloc[i:i + 1]); t_pred.append(time.perf_counter() - t0)
    with tempfile.NamedTemporaryFile(suffix='.pkl') as tmp:
        joblib.dump(m, tmp.name); ukuran = os.path.getsize(tmp.name) / 1024 ** 2
    baris_sistem.append({'Model': NAMA[key], 'Ekstraksi fitur (ms/citra)': 1000 * np.mean(t_fitur),
                         'Prediksi (ms/citra)': 1000 * np.mean(t_pred),
                         'Total latensi (ms/citra)': 1000 * (np.mean(t_fitur) + np.mean(t_pred)),
                         'Ukuran model (MB)': ukuran})
tabel_sistem = pd.DataFrame(baris_sistem).set_index('Model')
tabel_sistem.to_csv(os.path.join(SAVE_DIR, 'kinerja_sistem.csv'))
display(tabel_sistem.round(2))

lingkungan = {'Python': platform.python_version(), 'scikit-learn': sklearn.__version__,
              'scikit-image': skimage.__version__, 'NumPy': np.__version__, 'pandas': pd.__version__,
              'SciPy': scipy.__version__, 'OS': platform.platform(), 'Prosesor': platform.processor() or platform.machine(),
              'Jumlah inti CPU': os.cpu_count()}
try:
    lingkungan['RAM (GB)'] = round(os.sysconf('SC_PAGE_SIZE') * os.sysconf('SC_PHYS_PAGES') / 1024 ** 3, 1)
    lingkungan['Nama CPU'] = next(l.split(':')[1].strip() for l in open('/proc/cpuinfo') if 'model name' in l)
except Exception:
    pass
with open(os.path.join(SAVE_DIR, 'lingkungan_komputasi.json'), 'w') as f:
    json.dump(lingkungan, f, indent=2)
for k, v in lingkungan.items():
    print(f'{k:16s}: {v}')

## 11. Simpan model
Model skenario utama (GLCM+HSV) disimpan dalam format **`.pkl`** (joblib). Format `.h5` hanya untuk model Keras/TensorFlow, jadi tidak dipakai di sini.

In [ ]:
import joblib, sklearn

model_dir = os.path.join(APP_DIR, 'models')
for key in NAMA:
    joblib.dump(model_terbaik[UTAMA, key], os.path.join(model_dir, f'{key}_model.pkl'))

config = {
    'class_names': KELAS,
    'class_display': NAMA_KELAS,
    'feature_set': UTAMA,
    'feature_names': SKENARIO[UTAMA],
    'img_size': glcm_features.IMG_SIZE,
    'glcm_levels': glcm_features.LEVELS,
    'glcm_distances': glcm_features.DISTANCES,
    'best_params': {k: {p: (v if isinstance(v, (int, float, str)) or v is None else str(v))
                        for p, v in grid[UTAMA, k].best_params_.items()} for k in NAMA},
    'test_metrics': {k: {m: float(tabel_hasil.loc[(UTAMA, NAMA[k]), m]) for m in metrik} for k in NAMA},
    'sklearn_version': sklearn.__version__,
    'seed': SEED,
}
with open(os.path.join(model_dir, 'config.json'), 'w', encoding='utf-8') as f:
    json.dump(config, f, indent=2, ensure_ascii=False)

for f in sorted(os.listdir(model_dir)):
    print(f'{f:20s} {os.path.getsize(os.path.join(model_dir, f)) / 1e6:7.2f} MB')

In [ ]:
# Uji model yang sudah disimpan pada satu gambar uji acak (alur sama seperti web demo)
muat = {k: joblib.load(os.path.join(model_dir, f'{k}_model.pkl')) for k in NAMA}
idx = X_test.sample(1, random_state=SEED).index[0]
path = df.loc[idx, 'path']
f, mask, ok = extract_features(path)
x = pd.DataFrame([f], columns=FITUR_SEMUA)[config['feature_names']]
print('File       :', df.loc[idx, 'file'])
print('Label asli :', NAMA_KELAS[df.loc[idx, 'label']])
for k, m in muat.items():
    print(f'{NAMA[k]:20s}: {NAMA_KELAS[m.predict(x)[0]]}  (keyakinan {m.predict_proba(x).max():.1%})')
plt.imshow(load_rgb(path)); plt.axis('off'); plt.show()

## 12. Paket web demo (Streamlit)
Sel berikut menulis `app.py`, `requirements.txt`, dan `README.md`, lalu membungkus seluruh folder web app (termasuk model) menjadi **`web_app_jagung.zip`** di Google Drive.

In [ ]:
%%writefile /content/web_app_jagung/app.py
"""
Web demo: Klasifikasi Penyakit Daun Jagung (fitur GLCM + HSV, K-NN & Random Forest)
Jalankan:  streamlit run app.py
"""
import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import streamlit as st
from PIL import Image

from glcm_features import all_feature_names, extract_features, load_rgb

BASE = Path(__file__).parent
MODEL_DIR = BASE / 'models'
NAMA_MODEL = {'knn': 'K-Nearest Neighbor', 'rf': 'Random Forest'}

st.set_page_config(page_title='Penyakit Daun Jagung', page_icon='🌽', layout='wide')


@st.cache_resource
def muat_model():
    cfg = json.loads((MODEL_DIR / 'config.json').read_text(encoding='utf-8'))
    models = {key: joblib.load(MODEL_DIR / f'{key}_model.pkl') for key in NAMA_MODEL}
    return cfg, models


cfg, models = muat_model()
NAMA_KELAS = cfg.get('class_display', {})

st.title('🌽 Klasifikasi Penyakit Daun Jagung')
st.write('Ekstraksi fitur tekstur **GLCM** dan warna **HSV**, lalu klasifikasi dengan '
         '**K-Nearest Neighbor** dan **Random Forest**.')

with st.sidebar:
    st.header('Tentang model')
    st.write('Kelas yang dikenali:')
    for c in cfg['class_names']:
        st.markdown(f'- {NAMA_KELAS.get(c, c)}')
    st.write(f"Fitur: {cfg.get('feature_set', 'GLCM')} ({len(cfg['feature_names'])} fitur)")
    st.subheader('Hasil pada data uji')
    hasil = pd.DataFrame(cfg['test_metrics']).T
    hasil.index = [NAMA_MODEL.get(i, i) for i in hasil.index]
    st.dataframe(hasil.map(lambda v: f'{v:.2%}'))

file = st.file_uploader('Unggah foto satu helai daun jagung (sebaiknya berlatar polos)',
                        type=['jpg', 'jpeg', 'png', 'bmp', 'webp'])
if file is None:
    st.info('Silakan unggah gambar untuk mulai memprediksi.')
    st.stop()

try:
    img = Image.open(file)
    img.load()                                    # paksa membaca seluruh isi file
    fitur, mask, ok = extract_features(img)
    rgb = load_rgb(img)
except Exception:
    st.error('File tidak dapat dibaca sebagai gambar. Unggah file JPG/PNG yang valid.')
    st.stop()
overlay = rgb.copy()
overlay[~mask] = (overlay[~mask] * 0.25).astype(np.uint8)

kiri, kanan = st.columns(2)
kiri.image(rgb, caption='Citra masukan (256 x 256)')
kanan.image(overlay, caption='Area daun hasil segmentasi' if ok
            else 'Segmentasi gagal, seluruh citra dipakai')

X = pd.DataFrame([fitur], columns=all_feature_names())[cfg['feature_names']]
st.subheader('Hasil prediksi')
kolom = st.columns(len(models))
for k, (key, model) in zip(kolom, models.items()):
    proba = model.predict_proba(X)[0]
    kelas = [NAMA_KELAS.get(str(c), str(c)) for c in model.classes_]
    i = int(np.argmax(proba))
    k.markdown(f'#### {NAMA_MODEL[key]}')
    k.metric('Prediksi', kelas[i])
    k.caption(f'Keyakinan: {proba[i]:.1%}')
    k.bar_chart(pd.DataFrame({'Probabilitas': proba}, index=kelas))

with st.expander('Lihat nilai fitur gambar ini'):
    st.dataframe(X.T.rename(columns={0: 'Nilai'}))

In [ ]:
import skimage, scipy
req = [
    'streamlit',
    f'scikit-learn=={sklearn.__version__}',   # versi HARUS sama dengan saat melatih model
    f'numpy=={np.__version__.split(".")[0]}.*',
    'scikit-image', 'scipy', 'pandas', 'pillow', 'joblib',
]
with open(os.path.join(APP_DIR, 'requirements.txt'), 'w') as f:
    f.write('\n'.join(req) + '\n')

py_ver = f'{sys.version_info.major}.{sys.version_info.minor}'
readme = f"""# Web Demo Klasifikasi Penyakit Daun Jagung (GLCM + HSV, K-NN & Random Forest)

## Menjalankan di komputer sendiri (Windows)
1. Pasang **Python {py_ver}** dari https://www.python.org/downloads/
   (versi ini SAMA dengan Google Colab saat model dilatih; centang "Add python.exe to PATH").
   Versi Python yang lebih baru bisa gagal memasang scikit-learn=={sklearn.__version__}.
2. Ekstrak zip ini, buka foldernya di File Explorer, klik address bar, ketik cmd, tekan Enter.
3. Jalankan:
       py -{py_ver} -m pip install -r requirements.txt
       py -{py_ver} -m streamlit run app.py
4. Browser akan terbuka di http://localhost:8501 (tutup dengan Ctrl + C di terminal).

Mac/Linux: pakai python{py_ver} -m pip ... dan python{py_ver} -m streamlit run app.py

## Online gratis (Streamlit Community Cloud)
1. Unggah isi folder ini ke repository GitHub.
2. Buka https://share.streamlit.io, pilih repository, file utama: app.py,
   pada Advanced settings pilih Python {py_ver}, lalu Deploy.

Isi folder: app.py (tampilan web), glcm_features.py (preprocessing + fitur GLCM & HSV, sama dengan saat melatih),
models/ (knn_model.pkl, rf_model.pkl, config.json), requirements.txt.
"""
with open(os.path.join(APP_DIR, 'README.md'), 'w', encoding='utf-8') as f:
    f.write(readme)

shutil.rmtree(os.path.join(APP_DIR, '__pycache__'), ignore_errors=True)
zip_path = shutil.make_archive(os.path.join(SAVE_DIR, 'web_app_jagung'), 'zip', APP_DIR)
print('Web app tersimpan di:', zip_path)
print('\nSemua hasil di', SAVE_DIR, ':')
for f in sorted(os.listdir(SAVE_DIR)):
    print('  ', f)

## Catatan untuk artikel
- **Metodologi**: jumlah data awal (`tabel_dataset_awal.csv`) & hasil pembersihan (`tabel_pembersihan_data.csv`), segmentasi daun berbasis HSV, parameter GLCM, pembagian 80:20 ke folder latih/uji (`tabel_pembagian_data.csv`, daftar file di `daftar_pembagian_data.csv`), seed 42, 5-fold CV, dan ruang parameter GridSearch.
- **Hasil**: `tabel_perbandingan_model.csv` (2 skenario fitur × 2 algoritma), `confusion_matrix.png`, `perbandingan_metrik.png`.
- **Pembahasan**: kelas yang sering tertukar (biasanya bercak kelabu dan hawar daun karena lesinya sama-sama memanjang dan pucat), pengaruh nilai k (`pengaruh_nilai_k.png`), kontribusi fitur warna dibanding GLCM saja (selisih kedua skenario) dan fitur paling berpengaruh (`feature_importance_rf.png`), lalu bandingkan dengan Purba dkk. (2025) yang juga menemukan Random Forest lebih unggul dari K-NN.
- **Keterbatasan**: foto PlantVillage diambil dalam kondisi terkontrol (satu daun, latar polos), sehingga kinerja pada foto lapangan bisa lebih rendah; momen warna HSV peka terhadap pencahayaan.